In [134]:
#pip install -r requirements.txt

In [222]:
import time

In [135]:
import os
os.environ["LANGSMITH_TRACING"] = "false"

In [136]:


# Common test questions we'll use throughout:
TEST_QUESTIONS = [
    "What is an AI system under the EU AI Act?",
    "Does the EU AI Act apply to companies outside the EU?",
    "Who is considered a provider?",
    "What requirements apply to high-risk AI systems?",
    "How can a company become compliant with the AI Act?",
]

In [137]:
import helper.auth_utils as auth_utils,warnings
import numpy as np
from openai import OpenAI
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores.pgvector import PGVector, DistanceStrategy
from langchain.docstore.document import Document

In [138]:
# Your existing auth modules — same as LLM setup
import helper.cert_utils as cert_utils
from typing import List
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
load_dotenv()

True

In [139]:
load_dotenv()

True

In [140]:
class DSXEmbeddingInterface(Embeddings):
   

    def __init__(self, url, model_name, http_client, default_headers):
        self.client = OpenAI(
            base_url=url,
            http_client=http_client,
            api_key='not-empty',
            default_headers=default_headers
        )
        self.model_name = model_name

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """Embed search docs."""
        resp = self.client.embeddings.create(
            model=self.model_name,
            input=texts
        )
        return [a.embedding for a in resp.data]

    def embed_query(self, text: str) -> List[float]:
        """Embed query text."""
        return self.embed_documents([text])[0]

In [ ]:
PGVECTOR_CONNECTION_STRING = os.getenv("PGVECTOR_CONNECTION_STRING", "")
EMBEDDING_API_URL = os.getenv("EMBEDDING_API_URL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")

COLLECTION_CASES = "agentlens_case_descriptions"
COLLECTION_REMARKS = "agentlens_case_remarks"

BATCH_SIZE = 10
BATCH_DELAY = 2


In [142]:
_embeddings = None
_case_store = None
_remarks_store = None


def get_embeddings() -> DSXEmbeddingInterface:
    global _embeddings
    if _embeddings is None:
        # Same auth setup as your LLM in llm_setup.py
        settings = auth_utils.get_auth_settings('.env')
        auth_utils.ensure_credentials_if_needed(settings)
        cert_utils.update_dell_certificates()

        default_headers = auth_utils.build_default_headers(settings)
        http_client = auth_utils.build_http_client(settings)

        _embeddings = DSXEmbeddingInterface(
            url=EMBEDDING_API_URL,
            model_name=EMBEDDING_MODEL,
            http_client=http_client,
            default_headers=default_headers,
        )
        print(f"  🧠 Embedding model initialized: {EMBEDDING_MODEL}")
    return _embeddings



In [143]:
# Load libraries
import os
import sys
from dotenv import load_dotenv, find_dotenv
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import MessagesPlaceholder
from langchain.schema import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain, SequentialChain, ConversationChain
from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory, ConversationSummaryMemory
import re
from langchain.docstore.document import Document


In [144]:


from llm_setup import build_llm

llm, auth_components = build_llm()

print(f"✅ Dell LLM initialized successfully")

Initializing LLM (Approach 1)...
Model: llama-3-3-70b-instruct
Gateway: https://aia.gateway.dell.com/genai/dev/v1
Using Client Credentials
Downloaded certificate zip, size: 14933 bytes
Certifi bundle path: C:\Users\Prabal_Paul\AppData\Roaming\Python\Python312\site-packages\certifi\cacert.pem
Appending Dell certificates to certifi bundle...
Dell certificates successfully added to certifi bundle.
✅ LLM initialized successfully.
✅ Dell LLM initialized successfully


In [ ]:
from config import LLM_BASE_URL, LLM_MODEL, LLM_MAX_TOKENS

def get_dell_llm(temperature=0.3, max_tokens=None):
    """
    Helper to create additional LLM instances with different settings.
    Reuses the auth_components from the initial build_llm() call.
    """
    return ChatOpenAI(
        base_url=LLM_BASE_URL,
        model=LLM_MODEL,
        temperature=temperature,
        max_tokens=max_tokens or LLM_MAX_TOKENS,
        api_key="not-empty",
        default_headers=auth_components["default_headers"],
        http_client=auth_components["http_client"],
        streaming=False,
    )

print("✅  LLM helper function ready")

✅ Dell LLM helper function ready


In [146]:
from openai import OpenAI
from langchain_core.embeddings import Embeddings
from typing import List
import tiktoken

In [ ]:

class DSXEmbeddingInterface(Embeddings):


    # embedding gemma limit as confirmed by the 413 error
    MAX_TOKENS = 2000  # Slightly under 2048 for safety margin

    def __init__(self, url, model_name, http_client, default_headers):
        self.client = OpenAI(
            base_url=url,
            http_client=http_client,
            api_key='not-empty',
            default_headers=default_headers
        )
        self.model_name = model_name

        # Try to load a tokenizer for accurate counting
        # Fall back to character-based estimation if unavailable
        try:
            self._tokenizer = tiktoken.get_encoding("cl100k_base")
            self._use_tiktoken = True
        except Exception:
            self._use_tiktoken = False
            print("  ⚠️ tiktoken not available, using char-based estimation (1 token ≈ 4 chars)")

    def _count_tokens(self, text: str) -> int:
        """Count tokens in text."""
        if self._use_tiktoken:
            return len(self._tokenizer.encode(text))
        else:
            # Conservative estimate: 1 token ≈ 4 characters
            return len(text) // 4

    def _truncate_text(self, text: str) -> str:
        """Truncate text to fit within token limit."""
        token_count = self._count_tokens(text)
        if token_count <= self.MAX_TOKENS:
            return text

        if self._use_tiktoken:
            tokens = self._tokenizer.encode(text)
            truncated_tokens = tokens[:self.MAX_TOKENS]
            return self._tokenizer.decode(truncated_tokens)
        else:
            # Character-based truncation (conservative)
            max_chars = self.MAX_TOKENS * 4
            return text[:max_chars]

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """
        Embed search docs — one at a time with truncation.
        Handles oversized chunks gracefully instead of failing the entire batch.
        """
        all_embeddings = []
        for i, text in enumerate(texts):
            original_tokens = self._count_tokens(text)
            truncated_text = self._truncate_text(text)

            if original_tokens > self.MAX_TOKENS:
                print(f"    ⚠️ Text {i} truncated: {original_tokens} → {self.MAX_TOKENS} tokens")

            try:
                resp = self.client.embeddings.create(
                    model=self.model_name,
                    input=[truncated_text]
                )
                all_embeddings.append(resp.data[0].embedding)
            except Exception as e:
                print(f"    ❌ Failed to embed text {i} ({self._count_tokens(truncated_text)} tokens): {e}")
                # Return zero vector as fallback so indexing doesn't break
                # We'll get the dimension from a successful embedding or use 1024 (embedding gemma default)
                dim = len(all_embeddings[0]) if all_embeddings else 1024
                all_embeddings.append([0.0] * dim)

        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        """Embed query text."""
        truncated_text = self._truncate_text(text)
        resp = self.client.embeddings.create(
            model=self.model_name,
            input=[truncated_text]
        )
        return resp.data[0].embedding


# ═══════════════════════════════════════════════════════════
# Initialize with same auth as before
# ═══════════════════════════════════════════════════════════

EMBEDDING_API_URL = os.getenv("EMBEDDING_API_URL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "embedding gemma")

embeddings = DSXEmbeddingInterface(
    url=EMBEDDING_API_URL,
    model_name=EMBEDDING_MODEL,
    http_client=auth_components["http_client"],
    default_headers=auth_components["default_headers"],
)

# Quick test
test_vector = embeddings.embed_query("EU_regulation test")
print(f"✅ Embedding model initialized: {EMBEDDING_MODEL}")
print(f"   Vector dimension: {len(test_vector)}")

Generating new token...

✅ Embedding model initialized: embeddinggemma-300m
   Vector dimension: 768


In [148]:
# Load libraries
import os
import openai
from dotenv import load_dotenv, find_dotenv
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import MessagesPlaceholder
from langchain.schema import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain, SequentialChain, ConversationChain
from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory, ConversationSummaryMemory
import re
from langchain.docstore.document import Document


In [149]:
from langchain.docstore.document import Document

In [150]:
_ = load_dotenv(find_dotenv())
openai.api_key = os.getenv('OPENAI_API_KEY')

In [151]:
#help('langchain.text_splitter')

In [152]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter

In [153]:
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter

In [154]:
from langchain.document_loaders import PyPDFLoader,PyMuPDFLoader,csv_loader

# PDF File Load

In [155]:
loader=PyMuPDFLoader("OJ_L_202401689_EN_TXT.pdf")

In [156]:
docs=loader.load()

In [157]:
import inspect

In [158]:
print(inspect.signature(PyMuPDFLoader.__init__))

(self, file_path: Union[str, pathlib.PurePath], *, password: Optional[str] = None, mode: Literal['single', 'page'] = 'page', pages_delimiter: str = '\n\x0c', extract_images: bool = False, images_parser: Optional[langchain_community.document_loaders.parsers.images.BaseImageBlobParser] = None, images_inner_format: Literal['text', 'markdown-img', 'html-img'] = 'text', extract_tables: Optional[Literal['csv', 'markdown', 'html']] = None, headers: Optional[dict] = None, extract_tables_settings: Optional[dict[str, Any]] = None, **kwargs: Any) -> None


In [159]:
#! pip install rapidocr-onnxruntime
#from rapidocr import RapidOCR

In [160]:
len(docs[0].page_content)

3599

In [161]:
print(docs[0].page_content)

REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 13 June 2024
laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and 
Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)
(Text with EEA relevance)
THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,
Having regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof,
Having regard to the proposal from the European Commission,
After transmission of the draft legislative act to the national parliaments,
Having regard to the opinion of the European Economic and Social Committee (1),
Having regard to the opinion of the European Central Bank (2),
Having regard to the opinion of the Committee of the Regions (3),
Acting in accordance with the ordinary legislative procedure (4),
Whereas:
(1)
The p

In [162]:
docs[0].metadata

{'producer': 'PDFlib+PDI 9.0.7p3 (C++/Win64)',
 'creator': 'Servigistics Arbortext Publishing Engine',
 'creationdate': '2024-07-11T14:47:17+02:00',
 'source': 'OJ_L_202401689_EN_TXT.pdf',
 'file_path': 'OJ_L_202401689_EN_TXT.pdf',
 'total_pages': 144,
 'format': 'PDF 1.7',
 'title': 'Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)Text with EEA relevance.',
 'author': 'Publications Office of the European Union L-2985 Luxembourg LUXEMBOURG',
 'subject': 'I Legislative acts',
 'keywords': 'ISSN 1977-0677',
 'moddate': '2024-07-11T15:55:28+02:00',
 'trapped': '',
 'modDate': "D:20240711155528+02'00'",
 'creationDate': "D:20240711144717+02'00'",
 'page': 0}

In [163]:
len(docs)

144

## Perform splitting
      Split the data in a logical manner (i.e. paragraph wise/article wise/chapter wise/token wise)

In [164]:
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter

In [165]:
import pandas as pd

In [166]:
df=pd.DataFrame()
for i in range(len(docs)):
    df.loc[i, 'content'] = docs[i].page_content
    df.loc[i, 'length'] = len(docs[i].page_content)
    

In [167]:
df.sort_values(by='length', ascending=False)  

,content,length
2,(9)\nHarmonised rules applicable to the placin...,6884.0
20,(72)\nTo address concerns related to opacity a...,6331.0
4,(16)\nThe notion of ‘biometric categorisation’...,6281.0
35,providers in the AI regulatory sandbox should ...,6135.0
7,(27)\nWhile the risk-based approach is the bas...,6060.0
...,...,...
125,ANNEX II\nList of criminal offences referred t...,953.0
136,Section C — Information to be submitted by dep...,936.0
132,ANNEX VI\nConformity assessment procedure base...,807.0
128,(b) AI systems intended to be used for influen...,493.0


In [168]:
# Merge all page content
full_text = "\n".join([p.page_content for p in docs])

In [169]:
len(full_text)

599774

In [170]:
# stop here

## Paragraph Split

In [171]:
from langchain.schema import Document

def split_paragraphs_from_docs(docs):
    """
    Splits text from LangChain Document objects into paragraphs with estimated page metadata.
    """
    # Merge all page content and track character offsets
    full_text = ""
    page_starts = []  # List of (char_index, page_number)

    for doc in docs:
        start_idx = len(full_text)
        full_text += doc.page_content + "\n"
        page_starts.append((start_idx, doc.metadata.get("page", -1)))  # fallback if page not found

    # Normalize line endings and split by double newline
    full_text = full_text.replace('\r\n', '\n').replace('\r', '\n')
    raw_paragraphs = [para.strip() for para in full_text.split("\n\n") if para.strip()]

    # Assign metadata by estimating page number
    paragraph_docs = []
    for para in raw_paragraphs:
        start_pos = full_text.find(para)
        page_num = page_starts[0][1]

        for i in range(len(page_starts) - 1):
            if page_starts[i][0] <= start_pos < page_starts[i + 1][0]:
                page_num = page_starts[i][1]
                break
        else:
            if start_pos >= page_starts[-1][0]:
                page_num = page_starts[-1][1]

        paragraph_docs.append(Document(
            page_content=para,
            metadata={
                "chunking_method": "paragraph",
                "source": "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679",
                "page": page_num
            }
        ))

    return paragraph_docs


In [172]:
paragraph_docs=split_paragraphs_from_docs(docs)    

In [173]:
#paragraph_docs = split_paragraphs_from_docs(docs)

In [174]:
len(paragraph_docs)

1

In [175]:
len(paragraph_docs[0].page_content)

599774

## Article split

In [176]:
def split_articles(text):
    """
    Splits text from LangChain Document objects into articles, each with estimated page metadata.
    """
    # Merge all page content and track character offsets
    full_text = ""
    page_starts = []  # (char_index, page_number)

    for doc in docs:
        start_idx = len(full_text)
        full_text += doc.page_content + "\n"
        page_starts.append((start_idx, doc.metadata.get("page", -1)))  # -1 fallback if page not found

    # Normalize line endings
    full_text = full_text.replace('\r\n', '\n').replace('\r', '\n')
    lines = full_text.split('\n')
    new_lines = []

    for i, line in enumerate(lines):
        stripped = line.strip()

        # Match only "Article 1" to "Article 99" (not TFEU), not ending with ';'
        if re.match(r'^Article\s+[1-9][0-9]?\b(?!\s*TFEU)', stripped, re.IGNORECASE):
            if not stripped.endswith(';'):
                next_line = lines[i + 1].strip() if i + 1 < len(lines) else ""
                if next_line:
                    new_lines.append("<<ARTICLE_SPLIT>>")  # Insert marker
        new_lines.append(line)

    # Split and clean
    split_text = "\n".join(new_lines)
    chunks = split_text.split("<<ARTICLE_SPLIT>>")
    articles = [chunk.strip() for chunk in chunks if chunk.strip().lower().startswith("article")]

    # Assign metadata by estimating page number from position
    article_docs = []
    for article in articles:
        start_pos = full_text.find(article)
        page_num = page_starts[0][1]

        for i in range(len(page_starts) - 1):
            if page_starts[i][0] <= start_pos < page_starts[i + 1][0]:
                page_num = page_starts[i][1]
                break
        else:
            if start_pos >= page_starts[-1][0]:
                page_num = page_starts[-1][1]

        article_docs.append(Document(
            page_content=article,
            metadata={
                "chunking_method": "article",
                "source": "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679",
                "page": page_num
            }
        ))

    return article_docs

In [177]:
articles=split_articles(docs)

In [178]:
len(articles)

122

In [179]:
articles[5].metadata

{'chunking_method': 'article',
 'source': 'https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679',
 'page': 50}

## Chapter Split

In [180]:
def split_chapters(text):
    """
    Split chapters from a list of LangChain Document objects (with page metadata),
    and return new Document chunks with page and custom metadata.
    """
    # Combine all page texts but keep track of the starting index of each page
    full_text = ""
    page_starts = []  # List of (start_index, page_number)
    
    for doc in docs:
        page_start_index = len(full_text)
        full_text += doc.page_content + "\n"
        page_starts.append((page_start_index, doc.metadata.get("page", -1)))  # fallback -1 if no page

    # Now split chapters
    full_text = full_text.replace('\r\n', '\n').replace('\r', '\n')
    lines = full_text.split('\n')
    new_lines = []

    for i, line in enumerate(lines):
        stripped = line.strip()
        if re.match(r'^CHAPTER\s+(?:[I]{1,3}|IV|V|VI|VII|VIII|IX|X|XI)\b', stripped, re.IGNORECASE):
            if not stripped.endswith(';'):
                next_line = lines[i + 1].strip() if i + 1 < len(lines) else ""
                if next_line:
                    new_lines.append("<<chapter_SPLIT>>")
        new_lines.append(line)

    split_text = "\n".join(new_lines)
    chapters = [chunk.strip() for chunk in split_text.split("<<chapter_SPLIT>>") if chunk.strip().lower().startswith("chapter")]

    # Assign metadata by estimating page number from index
    chapter_docs = []
    for chapter in chapters:
        # Estimate start index of chapter
        chapter_start = full_text.find(chapter)
        # Default to first page
        page_num = page_starts[0][1]
        for i in range(len(page_starts) - 1):
            if page_starts[i][0] <= chapter_start < page_starts[i + 1][0]:
                page_num = page_starts[i][1]
                break
        else:
            # If chapter start is beyond last known start
            if chapter_start >= page_starts[-1][0]:
                page_num = page_starts[-1][1]

        # Add metadata
        chapter_docs.append(Document(
            page_content=chapter,
            metadata={
                "chunking_method": "chapter",
                "source": "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679",
                "page": page_num
            }
        ))

    return chapter_docs


chapter_docs = split_chapters(full_text)



In [181]:
len(chapter_docs)

12

In [182]:
chapter_docs[0].metadata

{'chunking_method': 'chapter',
 'source': 'https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679',
 'page': 43}

## Tokenwise Split

In [183]:
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter

In [184]:
docs[0].metadata

{'producer': 'PDFlib+PDI 9.0.7p3 (C++/Win64)',
 'creator': 'Servigistics Arbortext Publishing Engine',
 'creationdate': '2024-07-11T14:47:17+02:00',
 'source': 'OJ_L_202401689_EN_TXT.pdf',
 'file_path': 'OJ_L_202401689_EN_TXT.pdf',
 'total_pages': 144,
 'format': 'PDF 1.7',
 'title': 'Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)Text with EEA relevance.',
 'author': 'Publications Office of the European Union L-2985 Luxembourg LUXEMBOURG',
 'subject': 'I Legislative acts',
 'keywords': 'ISSN 1977-0677',
 'moddate': '2024-07-11T15:55:28+02:00',
 'trapped': '',
 'modDate': "D:20240711155528+02'00'",
 'creationDate': "D:20240711144717+02'00'",
 'page': 0}

In [185]:
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

In [186]:
split_docs=[]
for doc in docs:
    page_text=doc.page_content
    page_metadata={
        "chunking_method":"text_splitter",
        "source":"https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689",
        "page": doc.metadata.get("page", -1)
    }
    chunk=text_splitter.create_documents([page_text],metadatas=[page_metadata])
    split_docs.extend(chunk)

In [187]:
#  Set up the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

# Split each page and retain page-level metadata
split_docs = []
for doc in docs:
    page_text = doc.page_content
    page_metadata = {
        "chunking_method": "text_splitter",
        "source": "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689",
        "page": doc.metadata.get("page", -1)
    }                

    # Split text from each page and assign page metadata to each chunk
    chunks = text_splitter.create_documents([page_text], metadatas=[page_metadata])
    split_docs.extend(chunks)

In [188]:
len(split_docs)

775

In [189]:
split_docs[8].page_content

'Charter. As a prerequisite, AI should be a human-centric technology. It should serve as a tool for people, with the \nultimate aim of increasing human well-being.\n(7)\nIn order to ensure a consistent and high level of protection of public interests as regards health, safety and \nfundamental rights, common rules for high-risk AI systems should be established. Those rules should be consistent \nwith the Charter, non-discriminatory and in line with the Union’s international trade commitments. They should \nalso take into account the European Declaration on Digital Rights and Principles for the Digital Decade and the \nEthics guidelines for trustworthy AI of the High-Level Expert Group on Artificial Intelligence (AI HLEG).\n(8)\nA Union legal framework laying down harmonised rules on AI is therefore needed to foster the development, use \nand uptake of AI in the internal market that at the same time meets a high level of protection of public interests, such'

In [190]:
document=paragraph_docs+articles+chapter_docs+split_docs

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document

# Safe limit for embeddinggemma-300m
# ~1800 tokens * 4 chars/token = 7200, using 7000 for safety
MAX_CHUNK_CHARS = 7000

def preprocess_oversized_chunks(documents, max_chars=MAX_CHUNK_CHARS):
    """
    Sub-split any chunk exceeding the embedding model's token limit.
    Preserves ALL content by splitting into smaller pieces
    instead of truncating.
    
    Each sub-chunk inherits the parent's metadata with additional
    tracking fields (sub_chunk index, original_length).
    """
    sub_splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chars,
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    processed = []
    oversized_count = 0
    total_sub_chunks = 0

    for doc in documents:
        if len(doc.page_content) > max_chars:
            oversized_count += 1
            
            # Sub-split the oversized chunk
            sub_chunks = sub_splitter.create_documents(
                [doc.page_content],
                metadatas=[doc.metadata]
            )
            
            # Add tracking metadata to each sub-chunk
            for j, sub in enumerate(sub_chunks):
                sub.metadata = {
                    **doc.metadata,
                    "sub_chunk": j,
                    "total_sub_chunks": len(sub_chunks),
                    "original_length": len(doc.page_content),
                }
            
            processed.extend(sub_chunks)
            total_sub_chunks += len(sub_chunks)
        else:
            processed.append(doc)

    # Report
    print(f"  📏 Pre-processing complete:")
    print(f"     Original chunks:     {len(documents)}")
    print(f"     Oversized chunks:    {oversized_count}")
    print(f"     Sub-chunks created:  {total_sub_chunks}")
    print(f"     Final chunk count:   {len(processed)}")
    print(f"     Max chars per chunk: {max_chars} (~{max_chars // 4} tokens)")

    # Show which chunks were oversized
    if oversized_count > 0:
        print(f"\n  ⚠️ Oversized chunks that were split:")
        for doc in documents:
            if len(doc.page_content) > max_chars:
                method = doc.metadata.get('chunking_method', '?')
                print(f"     method={method}, "
                      f"chars={len(doc.page_content)}, "
                      f"~tokens={len(doc.page_content) // 4}")

    return processed


# ═══════════════════════════════════════════════════════════
# Show current document breakdown
# ═══════════════════════════════════════════════════════════

print("📦 Document breakdown BEFORE pre-processing:")
print(f"   paragraph_docs: {len(paragraph_docs)}")
print(f"   articles:       {len(articles)}")
print(f"   chapter_docs:   {len(chapter_docs)}")
print(f"   split_docs:     {len(split_docs)}")
print(f"   Total:          {len(document)}")

# ═══════════════════════════════════════════════════════════
# Pre-process: split oversized chunks
# ═══════════════════════════════════════════════════════════

document_processed = preprocess_oversized_chunks(document)

# ═══════════════════════════════════════════════════════════
# Now run indexing with pre-processed documents
# ═══════════════════════════════════════════════════════════

BATCH_SIZE = 5
BATCH_DELAY = 2

vs = index_EU_regulation_documents(document_processed, delete_existing=True)

# Embed and save it into a vector DB 

In [191]:
from langchain_community.vectorstores.pgvector import PGVector, DistanceStrategy

In [192]:
load_dotenv()

PGVECTOR_CONNECTION_STRING = os.getenv("PGVECTOR_CONNECTION_STRING", "")
COLLECTION_EU_regulation = "EU_regulation_regulations_new"  # Collection name for EU_regulation documents

BATCH_SIZE = 5
BATCH_DELAY = 2

if not PGVECTOR_CONNECTION_STRING:
    raise ValueError("⚠️ PGVECTOR_CONNECTION_STRING not set in .env")

print(f"✅ PGVector config loaded")
print(f"   Collection: {COLLECTION_EU_regulation}")


✅ PGVector config loaded
   Collection: EU_regulation_regulations_new


In [193]:
import time

### Index EU_regulation document into PGVector


def index_EU_regulation_documents(documents, delete_existing=False):
    """Index EU_regulation document chunks into PGVector with oversized chunk handling."""
    print(f"  📦 Indexing {len(documents)} EU_regulation chunks into {COLLECTION_EU_regulation}...")
    start = time.time()

    if not documents:
        print("  ⚠️ No documents to index")
        return None

    print(f"  📄 {len(documents)} documents ready")

    # First batch — create collection
    first_batch = documents[:BATCH_SIZE]
    try:
        store = PGVector.from_documents(
            embedding=embeddings,
            documents=first_batch,
            distance_strategy=DistanceStrategy.COSINE,
            collection_name=COLLECTION_EU_regulation,
            connection_string=PGVECTOR_CONNECTION_STRING,
            pre_delete_collection=delete_existing,
        )
        print(f"  📦 Indexed first batch: {len(first_batch)}/{len(documents)}")
    except Exception as e:
        print(f"  ⚠️ First attempt failed: {e}")
        print(f"  🔄 Retrying without pre_delete...")
        store = PGVector.from_documents(
            embedding=embeddings,
            documents=first_batch,
            distance_strategy=DistanceStrategy.COSINE,
            collection_name=COLLECTION_EU_regulation,
            connection_string=PGVECTOR_CONNECTION_STRING,
            pre_delete_collection=False,
        )

    # Remaining batches
    failed_batches = []
    for i in range(BATCH_SIZE, len(documents), BATCH_SIZE):
        batch = documents[i:i + BATCH_SIZE]
        try:
            store.add_documents(documents=batch)
            print(f"  📦 Indexed {i + len(batch)}/{len(documents)}")
            time.sleep(BATCH_DELAY)
            if i % (BATCH_SIZE * 10) == 0:
                elapsed = round(time.time() - start, 1)
                print(f"  ⏱️ {elapsed}s elapsed")
        except Exception as e:
            print(f"  ⚠️ Batch {i} failed: {e}")
            failed_batches.append(i)

            # Retry individual documents from failed batch
            print(f"  🔄 Retrying batch {i} one document at a time...")
            for j, doc in enumerate(batch):
                try:
                    store.add_documents(documents=[doc])
                except Exception as e2:
                    print(f"    ❌ Doc {i+j} permanently failed "
                          f"(chars={len(doc.page_content)}, "
                          f"method={doc.metadata.get('chunking_method','?')}): {e2}")

    elapsed = round(time.time() - start, 1)
    print(f"  ✅ Indexed {len(documents)} EU_regulation chunks in {elapsed}s")
    if failed_batches:
        print(f"  ⚠️ {len(failed_batches)} batches had failures (retried individually)")
    return store


# ═══════════════════════════════════════════════════════════
# Run indexing with PRE-PROCESSED documents
# ═══════════════════════════════════════════════════════════

vs = index_EU_regulation_documents(document_processed, delete_existing=True)



In [194]:
# ═══════════════════════════════════════════════════════════
# Inspect a specific chunk in detail
# ═══════════════════════════════════════════════════════════

def inspect_chunk(documents, index):
    """Show full details of a specific chunk."""
    if index >= len(documents):
        print(f"  ❌ Index {index} out of range (max: {len(documents)-1})")
        return
    
    doc = documents[index]
    print(f"{'=' * 70}")
    print(f"📋 CHUNK [{index}] — Full Inspection")
    print(f"{'=' * 70}")
    print(f"  Chunking method: {doc.metadata.get('chunking_method', 'N/A')}")
    print(f"  Page:            {doc.metadata.get('page', 'N/A')}")
    print(f"  Source:          {doc.metadata.get('source', 'N/A')}")
    
    # Show all metadata
    print(f"\n  📎 All metadata:")
    for key, value in doc.metadata.items():
        print(f"     {key}: {value}")
    
    print(f"\n  📏 Content length: {len(doc.page_content)} chars (~{len(doc.page_content)//3} tokens)")
    
    # Check if it's safe for embeddinggemma-300m
    if len(doc.page_content) > 5000:
        print(f"  🔴 WARNING: Exceeds embeddinggemma-300m safe limit (5000 chars)")
    else:
        print(f"  ✅ Within embeddinggemma-300m safe limit")
    
    print(f"\n  📝 Full content:")
    print(f"  {'─' * 60}")
    print(f"  {doc.page_content}")
    print(f"  {'─' * 60}")


# Examples — change the index to inspect different chunks
inspect_chunk(document_processed, 0)     # First paragraph chunk
inspect_chunk(document_processed, 100)   # Likely an article chunk
inspect_chunk(document_processed, 200)   # Another chunk

📋 CHUNK [0] — Full Inspection
  Chunking method: paragraph
  Page:            0
  Source:          https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679

  📎 All metadata:
     chunking_method: paragraph
     source: https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32016R0679
     page: 0
     sub_chunk: 0
     total_sub_chunks: 89
     original_length: 599774

  📏 Content length: 6921 chars (~2307 tokens)
  🔴 WARNING: Exceeds embeddinggemma-300m safe limit (5000 chars)

  📝 Full content:
  ────────────────────────────────────────────────────────────
  REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 13 June 2024
laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and 
Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)
(Text with EEA relevance)
THE EUROPEAN PARLIA

In [195]:
def get_EU_regulation_store():
    """Get or connect to the EU_regulation PGVector store."""
    store = PGVector(
        embedding_function=embeddings,
        collection_name=COLLECTION_EU_regulation,
        connection_string=PGVECTOR_CONNECTION_STRING,
    )
    print(f"  📦 EU_regulation vector store connected: {COLLECTION_EU_regulation}")
    return store


In [196]:
vs = get_EU_regulation_store()

  📦 EU_regulation vector store connected: EU_regulation_regulations_new


In [197]:
from langchain.chains import RetrievalQA

In [198]:
# Basic retriever for RAG chains
retriever = vs.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.4},
)

# Dedicated search function (following rag_store.py pattern)
def search_EU_regulation(query: str, k: int = 4, score_threshold: float = 0.4) -> list:
    """Search EU_regulation documents by semantic similarity."""
    EU_regulation_retriever = vs.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": k, "score_threshold": score_threshold},
    )
    results = EU_regulation_retriever.invoke(query)
    print(f"  🔍 EU_regulation search '{query[:50]}...' → {len(results)} results")
    return results

print("✅ Retriever created")

✅ Retriever created


In [199]:
user_query = 'What is an AI system under the EU AI Act?'

retrieved_chunks = search_EU_regulation(user_query, k=3)

print(f"✅ Retrieved {len(retrieved_chunks)} chunks")
for i, chunk in enumerate(retrieved_chunks):
    print(f"\nChunk {i+1} (method={chunk.metadata.get('chunking_method', 'N/A')}):")
    print(f"  {chunk.page_content[:150]}...")

Token not expired, using cached token...

  🔍 EU_regulation search 'What is an AI system under the EU AI Act?...' → 3 results
✅ Retrieved 3 chunks

Chunk 1 (method=text_splitter):
  the obligation to comply with this Regulation where an AI system falling into the scope of this Regulation is placed 
on the market or put into servic...

Chunk 2 (method=text_splitter):
  benefits, risks, safeguards, rights and obligations in relation to the use of AI systems. In cooperation with the relevant 
stakeholders, the Commissi...

Chunk 3 (method=text_splitter):
  (b) prohibitions of certain AI practices;
(c) specific requirements for high-risk AI systems and obligations for operators of such systems;
(d) harmon...


In [200]:
# Filtered search example by chunking method
filtered_results = vs.similarity_search(
    'What is the Processing of the national identification number?',
    k=3,
    filter={"chunking_method": "article"}
)

for i, doc in enumerate(filtered_results):
    print(f"Chunk {i+1}: {doc.page_content[:120]}...")

Token not expired, using cached token...

Chunk 1: within 24 hours. If such authorisation is rejected, the use shall be stopped with immediate effect and all the data, as ...
Chunk 2: Article 35
Identification numbers and lists of notified bodies
1.
The Commission shall assign a single identification nu...
Chunk 3: Article 78.
7.
Notifying authorities shall have an adequate number of competent personnel at their disposal for the prop...


In [201]:

def rag_chain(query):
    prompt = ChatPromptTemplate.from_template(
        """You are an agent. Do not respond to questions which leads to controversy or racism or adultry question. 
        .For SSN related query, reply with "Sorry i cannot respond to this"
        Chat History: {chat_history}

        context: {context}

        Question: {question}

        Answer:
        """
    )
    memory = ConversationBufferMemory(
        memory_key='chat_history',
        input_key='question'
    )

    chain = RetrievalQA.from_chain_type(
        llm=llm,  # ✅ Dell LLM (from build_llm())
        retriever=retriever,
        chain_type_kwargs={
            "prompt": prompt,
            "memory": memory
        }
    )

    return chain.invoke(query)

In [202]:
result = rag_chain("What is an AI system under the EU AI Act?")
print(result['result'])

Token not expired, using cached token...

Token not expired, using cached token...

The EU AI Act does not provide a explicit definition of an AI system. However, based on the context and the provisions outlined in the Act, an AI system can be understood as a software or program that uses artificial intelligence and machine learning to perform tasks, make decisions, or provide outputs. The Act applies to providers and deployers of AI systems, including those that place AI systems on the market, put them into service, or use them in the Union. The scope of the Act includes various types of AI systems, such as general-purpose AI models, high-risk AI systems, and AI systems used in different sectors and industries.


In [203]:
#Agentic RAG — Tools

from langchain.agents import initialize_agent, Tool, AgentType


def context_retriever(text):

    retrieved_docs = search_EU_regulation(text, k=3)
    return ("\n\n".join([doc.page_content for doc in retrieved_docs]))

Tool_retriever = Tool(
    name="Context Retriever",
    func=context_retriever,
    description=("retrieve regulations which are relevant for user's question/query."
            "Returns multiple relevant chunks. Use this to find specific "
        "articles, chapters, or regulatory text."
    )
)


def citation(text):
    llm_citation = get_dell_llm(temperature=0)
    citation_prompt = (
        "Find if any citation is in place. Keep in mind that citation is "
        "not article or paragraph or chapter. It is basically any reference."
    )
    try:
        return llm_citation.invoke(f"{citation_prompt} for given {text}").content
    except Exception as e:
        return f"Citation check failed: {e}"


Tool_citation = Tool(
    name="Citation Tool for text",
    func=citation,
    description="Provide citation for any given text"
)


def summarizer(text):

    llm_summary = get_dell_llm(temperature=0)
    return llm_summary.invoke(f"Summarize specific answer within 2 sentence for given {text}").content

Tool_summarizer = Tool(
    name="Answer Summarizer",
    func=summarizer,
    description="Summarize the data to make answer crisp and to the point"
)

print("✅ All tools updated to use Dell LLM + PGVector")


✅ All tools updated to use Dell LLM + PGVector


In [204]:
# Initialize Agent
tools = [Tool_retriever, Tool_citation, Tool_summarizer]

agent = initialize_agent(
    tools=tools,
    llm=llm,  
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=5,                # ← Prevent infinite loops
    early_stopping_method="generate", # ← Force a final answer when limit hit
    handle_parsing_errors=True,       # ← Handle format errors gracefully
    agent_kwargs={
        "prefix": """You are a EU_regulation regulatory assistant. Answer questions about EU_regulation 
regulations using the available tools. 

IMPORTANT RULES:
- Use the Context Retriever tool ONCE to find relevant regulations
- If the retrieved context doesn't answer the question, say so honestly
- Do NOT keep retrying the same search with different words
- After retrieving context, provide your Final Answer immediately
- For SSN related queries, reply with "Sorry I cannot respond to this"
- Do not respond to controversial, racist, or inappropriate questions

You have access to the following tools:""",
    }
)

print("✅ Agent initialized with  LLM")

✅ Agent initialized with  LLM


In [205]:
agent_result = agent.invoke(
    "What is an AI system under the EU AI Act?"
)
print(agent_result)



> Entering new AgentExecutor chain...
Token not expired, using cached token...

To answer this question, I need to find the relevant regulation in the EU AI Act that defines an AI system.

Action: Context Retriever
Action Input: EU AI Act definition of AI systemToken not expired, using cached token...

  🔍 EU_regulation search 'EU AI Act definition of AI system...' → 3 results

Observation: that the AI system concerned, when placed on the market or put into service, no longer presents that risk without undue 
delay, within a period it may prescribe.
EN
OJ L, 12.7.2024
108/144
ELI: http://data.europa.eu/eli/reg/2024/1689/oj

the obligation to comply with this Regulation where an AI system falling into the scope of this Regulation is placed 
on the market or put into service as a result of such research and development activity and to the application of 
provisions on AI regulatory sandboxes and testing in real world conditions. Furthermore, without prejudice to the 
exclusion of AI sy

In [206]:
import time

In [207]:
#Baseline RAG
def baseline_rag(query, k=5):
    """Simple retrieve-and-generate. No reranking, no query transformation."""
    start = time.time()

    # Step 1: Retrieve
    docs = vs.similarity_search(query, k=k)

    # Step 2: Build context
    context = "\n\n".join([doc.page_content for doc in docs])

    # Step 3: Generate
    prompt = f"""You are a EU_regulation regulatory expert. Answer ONLY based on the provided context.
If the context doesn't contain the answer, say "I don't have enough information."

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt).content
    elapsed = round(time.time() - start, 2)

    return {
        "answer": response,
        "num_docs": len(docs),
        "time": elapsed,
        "method": "baseline",
        "chunks_preview": [d.page_content[:80] + "..." for d in docs],
    }


# Run baseline for all test questions
print("=" * 70)
print("📊 BASELINE RAG RESULTS")
print("=" * 70)

baseline_results = {}
for q in TEST_QUESTIONS:
    result = baseline_rag(q)
    baseline_results[q] = result
    print(f"\n❓ {q}")
    print(f"⏱️  Time: {result['time']}s | Chunks: {result['num_docs']}")
    print(f"📝 Answer: {result['answer'][:600]}...")
    print("-" * 70)

📊 BASELINE RAG RESULTS
Token not expired, using cached token...

Token not expired, using cached token...


❓ What is an AI system under the EU AI Act?
⏱️  Time: 6.11s | Chunks: 5
📝 Answer: I don't have enough information. 

The provided context does not contain a definition of an AI system under the EU AI Act. It discusses the scope, application, and provisions of the Regulation, but does not provide a definition of what constitutes an AI system....
----------------------------------------------------------------------
Token not expired, using cached token...

Token not expired, using cached token...


❓ Does the EU AI Act apply to companies outside the EU?
⏱️  Time: 4.94s | Chunks: 5
📝 Answer: Yes, the EU AI Act applies to providers and deployers of AI systems that are established in a third country, to the extent the output produced by those systems is intended to be used in the Union....
----------------------------------------------------------------------
Token not expired, using

# Advanced Method Apply

In [214]:
import psycopg2
import time

def create_hnsw_index(connection_string):
    """
    Create HNSW index on PGVector embedding column.
    
    Uses UNQUALIFIED table names throughout to avoid
    'permission denied for schema' errors. PostgreSQL
    resolves the table through search_path automatically.
    """
    conn = psycopg2.connect(connection_string)
    conn.autocommit = True
    cur = conn.cursor()

    try:
        # ── Step 0: Detect which schema the table is in (for logging only) ──
        cur.execute("""
            SELECT schemaname
            FROM pg_tables
            WHERE tablename = 'langchain_pg_embedding'
            LIMIT 1;
        """)
        schema_result = cur.fetchone()
        schema = schema_result[0] if schema_result else "unknown"
        print(f"  📍 Table found in schema: {schema}")

        # ── Step 1: Check if HNSW index already exists ──
        cur.execute("""
            SELECT indexname, indexdef
            FROM pg_indexes
            WHERE tablename = 'langchain_pg_embedding'
            AND indexdef LIKE '%%hnsw%%';
        """)
        existing = cur.fetchall()
        if existing:
            print(f"  ℹ️ HNSW index already exists: {existing[0][0]}")
            return

        # ── Step 2: Detect embedding dimension ──
        # Uses UNQUALIFIED name — resolved through search_path
        cur.execute("""
            SELECT vector_dims(embedding)
            FROM langchain_pg_embedding
            LIMIT 1;
        """)
        result = cur.fetchone()
        if result is None:
            print("  ⚠️ No embeddings found in table.")
            return

        dimension = result[0]
        print(f"  📏 Detected embedding dimension: {dimension}")

        # ── Step 3: Check if column needs explicit dimensions ──
        cur.execute("""
            SELECT atttypmod
            FROM pg_attribute
            WHERE attrelid = 'langchain_pg_embedding'::regclass
            AND attname = 'embedding';
        """)
        typmod = cur.fetchone()

        if typmod and typmod[0] == -1:
            # Column has no explicit dimension — ALTER it
            print(f"  🔧 Altering embedding column to vector({dimension})...")
            cur.execute(f"""
                ALTER TABLE langchain_pg_embedding
                ALTER COLUMN embedding
                SET DATA TYPE vector({dimension});
            """)
            print(f"  ✅ Column altered to vector({dimension})")
        else:
            print(f"  ✅ Column already has explicit dimensions")

        # ── Step 4: Create HNSW index ──
        print("  🔨 Creating HNSW index...")
        start = time.time()

        cur.execute(f"""
            CREATE INDEX IF NOT EXISTS hnsw_eu_regulation_cosine_idx
            ON langchain_pg_embedding
            USING hnsw (embedding vector_cosine_ops)
            WITH (m = 16, ef_construction = 64);
        """)

        elapsed = round(time.time() - start, 1)
        print(f"  ✅ HNSW index created in {elapsed}s")

        # ── Step 5: ANALYZE for query planner ──
        cur.execute("ANALYZE langchain_pg_embedding;")
        print("  ✅ ANALYZE complete")

        # ── Step 6: Set ef_search ──
        # Try database-level first, fall back to session
        try:
            cur.execute("SELECT current_database();")
            db_name = cur.fetchone()[0]
            cur.execute(f'ALTER DATABASE "{db_name}" SET hnsw.ef_search = 40;')
            print("  ✅ hnsw.ef_search=40 set at DATABASE level")
        except Exception:
            cur.execute("SET hnsw.ef_search = 40;")
            print("  ✅ hnsw.ef_search=40 set at SESSION level")

    except Exception as e:
        print(f"  ⚠️ Error: {e}")
    finally:
        cur.close()
        conn.close()


def check_index_stats(connection_string):
    """
    Check index size and table stats.
    Uses UNQUALIFIED table names to avoid schema permission issues.
    """
    conn = psycopg2.connect(connection_string)
    cur = conn.cursor()

    try:
        # ── Detect schema (for display only) ──
        cur.execute("""
            SELECT schemaname
            FROM pg_tables
            WHERE tablename = 'langchain_pg_embedding'
            LIMIT 1;
        """)
        schema_result = cur.fetchone()
        schema = schema_result[0] if schema_result else "unknown"
        print(f"  📍 Table schema: {schema}")

        # ── List ALL indexes (no schema filter) ──
        cur.execute("""
            SELECT indexname, indexdef
            FROM pg_indexes
            WHERE tablename = 'langchain_pg_embedding'
            ORDER BY indexname;
        """)
        indexes = cur.fetchall()

        print(f"  📊 Indexes on langchain_pg_embedding:")
        if not indexes:
            print("     (no indexes found)")
        for name, indexdef in indexes:
            idx_type = "🟢 HNSW" if "hnsw" in indexdef.lower() else "⚪ Other"
            print(f"     {idx_type} | {name}")

        # ── Total embedding count (UNQUALIFIED name) ──
        cur.execute("SELECT COUNT(*) FROM langchain_pg_embedding;")
        count = cur.fetchone()[0]
        print(f"  📊 Total embeddings: {count}")

        # ── Check embedding dimension (UNQUALIFIED name) ──
        cur.execute("""
            SELECT vector_dims(embedding)
            FROM langchain_pg_embedding
            LIMIT 1;
        """)
        dim_result = cur.fetchone()
        if dim_result:
            print(f"  📏 Embedding dimension: {dim_result[0]}")

        # ── Check column type ──
        cur.execute("""
            SELECT CASE
                WHEN atttypmod = -1 THEN 'vector (no dimension)'
                ELSE 'vector(' || (atttypmod) || ')'
            END as col_type
            FROM pg_attribute
            WHERE attrelid = 'langchain_pg_embedding'::regclass
            AND attname = 'embedding';
        """)
        type_result = cur.fetchone()
        if type_result:
            print(f"  📊 Column type: {type_result[0]}")

        # ── Check ef_search setting ──
        try:
            cur.execute("SHOW hnsw.ef_search;")
            ef = cur.fetchone()
            if ef:
                print(f"  ⚙️ hnsw.ef_search: {ef[0]}")
        except Exception:
            pass

        # ── Verify HNSW ──
        hnsw_exists = any("hnsw" in idx[1].lower() for idx in indexes)
        if hnsw_exists:
            print("  ✅ HNSW index is ACTIVE — queries use ANN (approximate nearest neighbor)")
        else:
            print("  ⚠️ No HNSW index — queries use sequential scan (exact but slower)")

    except Exception as e:
        print(f"  ⚠️ Stats check failed: {e}")
    finally:
        cur.close()
        conn.close()


# ── Run ──
create_hnsw_index(PGVECTOR_CONNECTION_STRING)
check_index_stats(PGVECTOR_CONNECTION_STRING)

  📍 Table found in schema: aa_agentic_pgv
  ℹ️ HNSW index already exists: hnsw_gdpr_cosine_idx
  📍 Table schema: aa_agentic_pgv
  📊 Indexes on langchain_pg_embedding:
     ⚪ Other | _backfill_null_emb_langchain_pg_embedding
     🟢 HNSW | hnsw_eu_regulation_cosine_idx
     🟢 HNSW | hnsw_gdpr_cosine_idx
     ⚪ Other | idx_category
     ⚪ Other | idx_category_trgm
     ⚪ Other | idx_cmetadata_composite
     ⚪ Other | idx_cmetadata_table_name_pattern
     ⚪ Other | idx_commodity
     ⚪ Other | idx_commodity_trgm
     ⚪ Other | idx_company
     ⚪ Other | idx_company_trgm
     ⚪ Other | idx_content_hash
     ⚪ Other | idx_document_date
     ⚪ Other | idx_document_lower
     ⚪ Other | idx_document_trgm
     ⚪ Other | idx_embedding_categorization_with_collection
     ⚪ Other | idx_embedding_collection_filename
     ⚪ Other | idx_embedding_collection_id
     ⚪ Other | idx_embedding_custom_id
     ⚪ Other | idx_embedding_summaries_filename_collection
     ⚪ Other | idx_filename
     ⚪ Other | id

In [215]:
# ═══════════════════════════════════════════════════════════
# Compare: Before HNSW (exact) vs After HNSW (approximate)
# The answers should be nearly identical, but HNSW is faster
# ═══════════════════════════════════════════════════════════

print("=" * 70)
print("📊 FEATURE 1: HNSW ANN Index — Performance Comparison")
print("=" * 70)

for q in TEST_QUESTIONS[:]:
    result = baseline_rag(q)
    print(f"\n❓ {q}")
    print(f"⏱️  Time: {result['time']}s (with HNSW index)")
    print(f"📝 Answer: {result['answer'][:1000]}...")
    print(f"   Compare with baseline time: {baseline_results[q]['time']}s")
    print("-" * 70)

print("\n💡 NOTE: Speed improvement is more noticeable with larger datasets (10K+ docs).")
print("   With ~600 docs, the difference may be minimal.")

📊 FEATURE 1: HNSW ANN Index — Performance Comparison
Token not expired, using cached token...

Token not expired, using cached token...


❓ What is an AI system under the EU AI Act?
⏱️  Time: 6.56s (with HNSW index)
📝 Answer: I don't have enough information. 

The provided context does not contain a definition of what constitutes an "AI system" under the EU AI Act. It discusses the scope, application, and provisions of the Regulation, but does not provide a clear definition of an AI system....
   Compare with baseline time: 6.11s
----------------------------------------------------------------------
Token not expired, using cached token...

Token not expired, using cached token...


❓ Does the EU AI Act apply to companies outside the EU?
⏱️  Time: 5.72s (with HNSW index)
📝 Answer: Yes, the EU AI Act applies to providers and deployers of AI systems that are established in a third country, to the extent the output produced by those systems is intended to be used in the Union....
   Compa

In [218]:
#! pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [219]:
from sentence_transformers import CrossEncoder

In [220]:
# Load reranker model (runs locally, no API needed)
print("Loading cross-encoder reranker...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("✅ Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading cross-encoder reranker...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 16705.30it/s]


✅ Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [225]:
import time

In [229]:

def rag_with_reranking(query, top_k_retrieve=10, top_k_rerank=3):
    """
    Two-stage retrieval:
      Stage 1: Retrieve top_k_retrieve candidates (fast, bi-encoder via PGVector)
      Stage 2: Rerank to top_k_rerank using cross-encoder (slow, accurate)
    """
    start = time.time()

    # Stage 1: Broad retrieval — use plain similarity_search (no threshold)
    candidates = vs.similarity_search(query, k=top_k_retrieve)
    print(f"  📥 Retrieved {len(candidates)} candidates")

    # ── FIX: Early return includes ALL expected keys ──
    if not candidates:
        elapsed = round(time.time() - start, 2)
        return {
            "answer": "No relevant documents found.",
            "method": "reranked",
            "time": elapsed,
            "num_docs": 0,
            "reranked_scores": [],
        }

    # Stage 2: Cross-encoder reranking
    pairs = [[query, doc.page_content] for doc in candidates]
    scores = reranker.predict(pairs)

    # Sort by reranking score (descending)
    scored_docs = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )

    # Take top-k after reranking
    reranked_docs = [doc for doc, score in scored_docs[:top_k_rerank]]
    reranked_scores = [round(float(s), 4) for _, s in scored_docs[:top_k_rerank]]

    print(f"  🏆 Reranked scores: {reranked_scores}")

    # Show what changed
    original_top = [d.page_content[:60] for d in candidates[:top_k_rerank]]
    reranked_top = [d.page_content[:60] for d in reranked_docs]
    if original_top != reranked_top:
        print(f"  🔄 Reranking CHANGED the order!")
    else:
        print(f"  ✅ Reranking confirmed the original order")

    # Generate answer
    context = "\n\n".join([doc.page_content for doc in reranked_docs])
    prompt = f"""You are an EU AI Act regulatory expert. Answer ONLY based on the provided context.
If the context doesn't contain the answer, say "I don't have enough information."

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt).content
    elapsed = round(time.time() - start, 2)

    return {
        "answer": response,
        "num_docs": len(reranked_docs),
        "reranked_scores": reranked_scores,
        "time": elapsed,
        "method": "reranked",
    }


# ── Compare ──
print("=" * 70)
print("📊 FEATURE 2: Reranking — Comparison")
print("=" * 70)

for q in TEST_QUESTIONS[:3]:
    print(f"\n{'='*70}")
    print(f"❓ {q}")

    print(f"\n🔹 BASELINE:")
    b = baseline_results.get(q, baseline_rag(q))
    print(f"   Time: {b.get('time', 'N/A')}s")
    print(f"   Answer: {b.get('answer', 'N/A')[:200]}...")

    print(f"\n🔸 WITH RERANKING:")
    r = rag_with_reranking(q)
    print(f"   Time: {r.get('time', 'N/A')}s | Scores: {r.get('reranked_scores', [])}")
    print(f"   Answer: {r.get('answer', 'N/A')[:200]}...")

📊 FEATURE 2: Reranking — Comparison

❓ What is an AI system under the EU AI Act?

🔹 BASELINE:
Token not expired, using cached token...

Token not expired, using cached token...

   Time: 6.11s
   Answer: I don't have enough information. 

The provided context does not contain a definition of an AI system under the EU AI Act. It discusses the scope, application, and provisions of the Regulation, but do...

🔸 WITH RERANKING:
Token not expired, using cached token...

  📥 Retrieved 10 candidates
  🏆 Reranked scores: [4.0773, 3.5748, 3.4573]
  🔄 Reranking CHANGED the order!
Token not expired, using cached token...

   Time: 5.29s | Scores: [4.0773, 3.5748, 3.4573]
   Answer: I don't have enough information....

❓ Does the EU AI Act apply to companies outside the EU?

🔹 BASELINE:
Token not expired, using cached token...

Token not expired, using cached token...

   Time: 4.94s
   Answer: Yes, the EU AI Act applies to providers and deployers of AI systems that are established in a third count

In [ ]:
# Hybrid Search (Dense + Sparse BM25)

In [ ]:
#! pip install rank_bm25

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [230]:
from rank_bm25 import BM25Okapi

In [231]:

class HybridSearcher:
    """
    Combines PGVector dense retrieval with BM25 sparse retrieval.
    Uses Reciprocal Rank Fusion (RRF) to merge results.
    """

    def __init__(self, documents, vector_store, alpha=0.5):

        self.documents = documents
        self.vector_store = vector_store
        self.alpha = alpha

        # Build BM25 index from document contents
        print(f"  🔨 Building BM25 index from {len(documents)} documents...")
        tokenized_docs = [doc.page_content.lower().split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized_docs)
        print(f"  ✅ BM25 index built | alpha={alpha}")


    def search(self, query, k=5, rrf_k=60):

        dense_k = k * 3  # Retrieve more candidates for fusion
        sparse_k = k * 3

        # ── Dense retrieval (PGVector) ──
        dense_results = self.vector_store.similarity_search(query, k=dense_k)

        # ── Sparse retrieval (BM25) ──
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        top_sparse_indices = np.argsort(bm25_scores)[::-1][:sparse_k]

        # ── Reciprocal Rank Fusion ──
        doc_scores = {}

        # Score dense results by rank
        for rank, doc in enumerate(dense_results):
            doc_id = hash(doc.page_content[:100])
            if doc_id not in doc_scores:
                doc_scores[doc_id] = {"doc": doc, "dense_rank": rank, "sparse_rank": None, "score": 0}
            doc_scores[doc_id]["score"] += self.alpha * (1 / (rrf_k + rank))

        # Score sparse results by rank
        for rank, idx in enumerate(top_sparse_indices):
            if idx < len(self.documents):
                doc = self.documents[idx]
                doc_id = hash(doc.page_content[:100])
                if doc_id not in doc_scores:
                    doc_scores[doc_id] = {"doc": doc, "dense_rank": None, "sparse_rank": rank, "score": 0}
                doc_scores[doc_id]["score"] += (1 - self.alpha) * (1 / (rrf_k + rank))
                if doc_scores[doc_id]["sparse_rank"] is None:
                    doc_scores[doc_id]["sparse_rank"] = rank

        # Sort by fused score
        sorted_results = sorted(doc_scores.values(), key=lambda x: x["score"], reverse=True)

        # Log which source contributed
        top_results = sorted_results[:k]
        for i, item in enumerate(top_results):
            source = []
            if item["dense_rank"] is not None:
                source.append(f"dense(rank={item['dense_rank']})")
            if item["sparse_rank"] is not None:
                source.append(f"sparse(rank={item['sparse_rank']})")
            print(f"    Result {i+1}: score={item['score']:.4f} | {' + '.join(source)}")

        return [item["doc"] for item in top_results]


# Initialize hybrid searcher
hybrid_searcher = HybridSearcher(
    documents=document_processed, 
    vector_store=vs,
    alpha=0.6 
)



def rag_with_hybrid_search(query, k=3):
    """RAG using hybrid (dense + sparse) retrieval."""
    start = time.time()

    # Hybrid retrieval
    docs = hybrid_searcher.search(query, k=k)

    # Rerank the hybrid results (combine Feature 2 + 3)
    if len(docs) > 0:
        pairs = [[query, doc.page_content] for doc in docs]
        scores = reranker.predict(pairs)
        scored = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
        docs = [d for d, s in scored]

    context = "\n\n".join([doc.page_content for doc in docs])
    prompt = f"""You are a EU_regulation regulatory expert. Answer ONLY based on the provided context.
If the context doesn't contain the answer, say "I don't have enough information."

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt).content
    elapsed = round(time.time() - start, 2)

    return {
        "answer": response,
        "num_docs": len(docs),
        "time": elapsed,
        "method": "hybrid+reranked",
    }


# ── Compare ──
print("=" * 70)
print("📊 FEATURE 3: Hybrid Search — Comparison")
print("=" * 70)

# Test with a keyword-heavy query where BM25 should help
keyword_query = "Article 17 right to erasure right to be forgotten"

print(f"\n❓ {keyword_query}")

print(f"\n🔹 BASELINE (Dense only):")
b = baseline_rag(keyword_query)
print(f"   Answer: {b['answer'][:250]}...")

print(f"\n🔸 HYBRID (Dense + BM25 + Reranker):")
h = rag_with_hybrid_search(keyword_query)
print(f"   Answer: {h['answer'][:250]}...")

# Also test standard questions
for q in TEST_QUESTIONS[:2]:
    print(f"\n{'='*70}")
    print(f"❓ {q}")
    h = rag_with_hybrid_search(q)
    print(f"   Time: {h['time']}s")
    print(f"   Answer: {h['answer'][:200]}...")

  🔨 Building BM25 index from 1079 documents...
  ✅ BM25 index built | alpha=0.6
📊 FEATURE 3: Hybrid Search — Comparison

❓ Article 17 right to erasure right to be forgotten

🔹 BASELINE (Dense only):
Token not expired, using cached token...

Token not expired, using cached token...

   Answer: Article 17 of the General Data Protection Regulation (GDPR) states that data subjects have the right to erasure, also known as the "right to be forgotten". This means that individuals have the right to request that their personal data be erased witho...

🔸 HYBRID (Dense + BM25 + Reranker):
Token not expired, using cached token...

    Result 1: score=0.0129 | sparse(rank=0)
    Result 2: score=0.0066 | sparse(rank=1)
    Result 3: score=0.0065 | sparse(rank=2)
Token not expired, using cached token...

   Answer: I don't have enough information....

❓ What is an AI system under the EU AI Act?
Token not expired, using cached token...

    Result 1: score=0.0100 | dense(rank=0)
    Result 2: score=0.

In [ ]:
# Semantic Chunking

In [ ]:


class SemanticChunkerOptimized:
    """
    Semantic chunker optimized API:
    - Batches embedding calls (reduces API calls by 10-20x)
    - Respects 2048 token limit per text
    - Progress reporting
    - Timeout safety
    """

    MAX_TOKENS_PER_TEXT = 2000  # embedding gemma limit
    MAX_CHARS_PER_TEXT = 7500   # ~2000 tokens at 4 chars/token

    def __init__(self, embedding_model, breakpoint_percentile=75,
                 min_chunk_size=100, batch_size=20):
        """
        Args:
            embedding_model: DSXEmbeddingInterface instance
            breakpoint_percentile: Split where similarity drops below 
                                    this percentile (higher = fewer splits)
            min_chunk_size: Minimum characters per chunk
            batch_size: Number of sentences to embed per API call
                        (embedding gemma allows multiple texts per call)
        """
        self.embedding_model = embedding_model
        self.breakpoint_percentile = breakpoint_percentile
        self.min_chunk_size = min_chunk_size
        self.batch_size = batch_size

    def _split_sentences(self, text):
        """Split text into sentences."""
        sentences = re.split(r'(?<=[.!?])\s+', text)
        return [s.strip() for s in sentences if len(s.strip()) > 10]

    def _cosine_similarity(self, a, b):
        """Compute cosine similarity between two vectors."""
        a = np.array(a)
        b = np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

    def _batch_embed(self, texts):
        """
        Embed texts in batches instead of one-by-one.
        Each text is truncated to stay under embedding gemma token limit.
        
        Instead of N API calls, this makes N/batch_size calls.
        """
        all_embeddings = []
        total = len(texts)

        for i in range(0, total, self.batch_size):
            batch = texts[i:i + self.batch_size]

            # Truncate each text in the batch
            truncated_batch = [
                t[:self.MAX_CHARS_PER_TEXT] for t in batch
            ]

            try:
                # Single API call for entire batch
                batch_embeddings = self.embedding_model.embed_documents(truncated_batch)
                all_embeddings.extend(batch_embeddings)
            except Exception as e:
                print(f"    ⚠️ Batch {i}-{i+len(batch)} failed: {e}")
                # Fallback: try one at a time
                for t in truncated_batch:
                    try:
                        emb = self.embedding_model.embed_query(t)
                        all_embeddings.append(emb)
                    except Exception:
                        dim = len(all_embeddings[0]) if all_embeddings else 1024
                        all_embeddings.append([0.0] * dim)

            if (i + self.batch_size) % (self.batch_size * 5) == 0 or i + self.batch_size >= total:
                print(f"    📊 Embedded {min(i + self.batch_size, total)}/{total} sentences")

        return all_embeddings

   
    def split_text(self, text, metadata=None):
        """Split text into semantically coherent chunks."""
        start = time.time()

        # Step 1: Split into sentences
        sentences = self._split_sentences(text)
        print(f"    📝 {len(sentences)} sentences extracted")

        if len(sentences) <= 1:
            return [Document(page_content=text, metadata=metadata or {})]

        # Step 2: Batch embed all sentences
        print(f"    🧠 Embedding sentences in batches of {self.batch_size}...")
        sentence_embeddings = self._batch_embed(sentences)
        embed_time = round(time.time() - start, 1)
        print(f"    ⏱️ Embedding completed in {embed_time}s")

        # Step 3: Compute similarities between consecutive sentences
        similarities = []
        for i in range(len(sentence_embeddings) - 1):
            sim = self._cosine_similarity(
                sentence_embeddings[i],
                sentence_embeddings[i + 1]
            )
            similarities.append(sim)

        if not similarities:
            return [Document(page_content=text, metadata=metadata or {})]

        # Step 4: Find breakpoints
        threshold = np.percentile(similarities, 100 - self.breakpoint_percentile)
        print(f"    📏 Similarity threshold: {threshold:.4f} "
              f"(percentile={self.breakpoint_percentile})")
        print(f"    📊 Similarity range: min={min(similarities):.4f}, "
              f"max={max(similarities):.4f}, mean={np.mean(similarities):.4f}")

        # Step 5: Group sentences into chunks at breakpoints
        chunks = []
        current_chunk = [sentences[0]]

        for i, sim in enumerate(similarities):
            if sim < threshold and len(" ".join(current_chunk)) >= self.min_chunk_size:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i + 1]]
            else:
                current_chunk.append(sentences[i + 1])

        if current_chunk:
            chunks.append(" ".join(current_chunk))

        # Step 6: Convert to Documents
        documents = []
        for i, chunk_text in enumerate(chunks):
            doc_metadata = {
                **(metadata or {}),
                "chunk_type": "semantic",
                "semantic_chunk_idx": i,
            }
            documents.append(Document(page_content=chunk_text, metadata=doc_metadata))

        elapsed = round(time.time() - start, 1)
        print(f"    ✅ Created {len(documents)} semantic chunks in {elapsed}s")
        return documents


# ═══════════════════════════════════════════════════════════
# Initialize and run
# ═══════════════════════════════════════════════════════════

semantic_chunker = SemanticChunkerOptimized(
    embedding_model=embeddings,
    breakpoint_percentile=75,
    min_chunk_size=100,
    batch_size=20   # 20 sentences per API call instead of 1
)

# ─────────────────────────────────────────────────────────
# IMPORTANT: Use pages[:5] (actual PDF pages, ~4K chars each)
# NOT document[:5] (which may include huge chapter_docs)
# ─────────────────────────────────────────────────────────

print("📦 Creating semantic chunks from first 5 PDF pages...")
sample_text = "\n".join([p.page_content for p in document[:2]])  # ← Use 'pages' not 'document'
print(f"   Sample text length: {len(sample_text)} chars")

semantic_chunks = semantic_chunker.split_text(
    sample_text,
    metadata={"chunking_method": "semantic", "source": "EU_regulation"}
)

# Compare chunk quality
print(f"\n📊 Chunking Comparison (first 5 pages):")
recursive_count = len([
    d for d in document_processed
    if d.metadata.get('page', 999) < 5
])
print(f"   Recursive splitter:  {recursive_count} chunks")
print(f"   Semantic splitter:   {len(semantic_chunks)} chunks")

print(f"\n📌 Sample Semantic Chunks:")
for i, chunk in enumerate(semantic_chunks[:3]):
    print(f"\n   Chunk {i+1} ({len(chunk.page_content)} chars):")
    print(f"   {chunk.page_content[:150]}...")

📦 Creating semantic chunks from first 5 PDF pages...
   Sample text length: 747982 chars
    📝 2598 sentences extracted
    🧠 Embedding sentences in batches of 20...
Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached toke

In [233]:
# Index semantic chunks into a SEPARATE collection for comparison
COLLECTION_EU_regulation_SEMANTIC = "EU_regulation_semantic_chunks"

# Only if you have enough semantic chunks to be useful
if len(semantic_chunks) >= 5:
    first_batch = semantic_chunks[:BATCH_SIZE]
    vs_semantic = PGVector.from_documents(
        embedding=embeddings,
        documents=first_batch,
        distance_strategy=DistanceStrategy.COSINE,
        collection_name=COLLECTION_EU_regulation_SEMANTIC,
        connection_string=PGVECTOR_CONNECTION_STRING,
        pre_delete_collection=True,
    )
    for i in range(BATCH_SIZE, len(semantic_chunks), BATCH_SIZE):
        batch = semantic_chunks[i:i + BATCH_SIZE]
        vs_semantic.add_documents(batch)
        time.sleep(BATCH_DELAY)

    print(f"✅ Semantic chunks indexed: {len(semantic_chunks)}")

    # Compare retrieval
    q = TEST_QUESTIONS[0]
    print(f"\n❓ {q}")
    print(f"\n🔹 RECURSIVE chunks retrieval:")
    for d in vs.similarity_search(q, k=2):
        print(f"   {d.page_content[:120]}...")
    print(f"\n🔸 SEMANTIC chunks retrieval:")
    for d in vs_semantic.similarity_search(q, k=2):
        print(f"   {d.page_content[:120]}...")

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...



Collection not found


Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached to

KeyboardInterrupt: 

In [ ]:
# Parent-Child (Hierarchical) Chunking
# ═══════════════════════════════════════════════════════════

from langchain.text_splitter import RecursiveCharacterTextSplitter

def create_parent_child_index(pages, parent_size=6000, child_size=1000, child_overlap=150):
    """
    Create two-tier chunks:
      - Parents: large chunks for LLM context
      - Children: small chunks for search (mapped to parents)

    Both respect the embedding gemma 2048 token limit (~7500 chars)
    """
    # Create parent chunks
    parent_splitter = RecursiveCharacterTextSplitter(
        chunk_size=parent_size,
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " "]
    )
    parent_chunks = parent_splitter.split_documents(pages)
    print(f"  📦 Created {len(parent_chunks)} parent chunks (size={parent_size})")

    # Create child chunks from each parent
    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=child_size,
        chunk_overlap=child_overlap,
        separators=["\n\n", "\n", ". ", " "]
    )

    all_children = []
    parent_map = {}  # child_id → parent_doc

    for parent_idx, parent in enumerate(parent_chunks):
        children = child_splitter.split_text(parent.page_content)
        for child_idx, child_text in enumerate(children):
            child_doc = Document(
                page_content=child_text,
                metadata={
                    **parent.metadata,
                    "parent_idx": parent_idx,
                    "child_idx": child_idx,
                    "chunk_type": "child",
                }
            )
            all_children.append(child_doc)
            parent_map[f"{parent_idx}_{child_idx}"] = parent

    print(f"  📦 Created {len(all_children)} child chunks (size={child_size})")
    print(f"  📊 Avg children per parent: {len(all_children) / max(len(parent_chunks), 1):.1f}")

    return all_children, parent_chunks, parent_map


# Build parent-child hierarchy
children, parents, parent_map = create_parent_child_index(docs)

# Index CHILDREN into PGVector (used for search)
COLLECTION_EU_regulation_CHILDREN = "EU_regulation_child_chunks"

print(f"\n📦 Indexing {len(children)} child chunks...")
first_batch = children[:BATCH_SIZE]
vs_children = PGVector.from_documents(
    embedding=embeddings,
    documents=first_batch,
    distance_strategy=DistanceStrategy.COSINE,
    collection_name=COLLECTION_EU_regulation_CHILDREN,
    connection_string=PGVECTOR_CONNECTION_STRING,
    pre_delete_collection=True,
)

for i in range(BATCH_SIZE, len(children), BATCH_SIZE):
    batch = children[i:i + BATCH_SIZE]
    try:
        vs_children.add_documents(batch)
        if i % (BATCH_SIZE * 5) == 0:
            print(f"  📦 Indexed {i + len(batch)}/{len(children)}")
        time.sleep(BATCH_DELAY)
    except Exception as e:
        print(f"  ⚠️ Batch {i} failed: {e}")

print(f"✅ Child chunks indexed: {len(children)}")

  📦 Created 88 parent chunks (size=6000)
  📦 Created 455 child chunks (size=1000)
  📊 Avg children per parent: 5.2

📦 Indexing 455 child chunks...
Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...



Collection not found


Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached to

In [ ]:

def rag_parent_child(query, k_children=5, k_parents=3):
    """
    Search children (precise) → fetch parents (rich context) → generate.
    """
    start = time.time()

    # Step 1: Search children
    child_results = vs_children.similarity_search(query, k=k_children)
    print(f"  🔍 Found {len(child_results)} matching children")

    # Step 2: Map children → parents (deduplicate)
    seen_parents = set()
    parent_docs = []
    for child in child_results:
        pidx = child.metadata.get("parent_idx")
        if pidx is not None and pidx not in seen_parents:
            seen_parents.add(pidx)
            if pidx < len(parents):
                parent_docs.append(parents[pidx])

    parent_docs = parent_docs[:k_parents]
    print(f"  📄 Mapped to {len(parent_docs)} unique parents")

    # Step 3: Generate with parent context
    context = "\n\n".join([p.page_content for p in parent_docs])
    prompt = f"""You are a EU_regulation regulatory expert. Answer ONLY based on the provided context.
If the context doesn't contain the answer, say "I don't have enough information."

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt).content
    elapsed = round(time.time() - start, 2)

    return {
        "answer": response,
        "children_found": len(child_results),
        "parents_used": len(parent_docs),
        "time": elapsed,
        "method": "parent_child",
    }


# ── Compare ──
print("=" * 70)
print("📊 FEATURE 5: Parent-Child Chunking — Comparison")
print("=" * 70)

for q in TEST_QUESTIONS[:3]:
    print(f"\n{'='*70}")
    print(f"❓ {q}")

    print(f"\n🔹 BASELINE (flat chunks):")
    b = baseline_rag(q)
    print(f"   Answer: {b['answer'][:200]}...")

    print(f"\n🔸 PARENT-CHILD:")
    pc = rag_parent_child(q)
    print(f"   Children→Parents: {pc['children_found']}→{pc['parents_used']} | Time: {pc['time']}s")
    print(f"   Answer: {pc['answer'][:200]}...")

📊 FEATURE 5: Parent-Child Chunking — Comparison

❓ What is the regulation policy for children under 10 Years?

🔹 BASELINE (flat chunks):
Token not expired, using cached token...

Token not expired, using cached token...

   Answer: I don't have enough information....

🔸 PARENT-CHILD:
Token not expired, using cached token...

  🔍 Found 5 matching children
  📄 Mapped to 3 unique parents
Token not expired, using cached token...

   Children→Parents: 5→3 | Time: 5.65s
   Answer: I don't have enough information....

❓ What are the conditions for lawful processing of personal data?

🔹 BASELINE (flat chunks):
Token not expired, using cached token...

Token not expired, using cached token...

   Answer: The context mentions several conditions for lawful processing of personal data, including: 

(a) consent of the data subject 
(b) appropriate safeguards for the fundamental rights and the interests of...

🔸 PARENT-CHILD:
Token not expired, using cached token...

  🔍 Found 5 matching children
  

In [ ]:
# HyDE (Hypothetical Document Embeddings)

In [235]:
def rag_with_hyde(query, k=3):
    """
    HyDE RAG:
    1. Generate hypothetical answer using LLM
    2. Embed the hypothetical answer
    3. Retrieve using hypothetical embedding
    4. Generate final answer with retrieved context
    """
    start = time.time()

    # Step 1: Generate hypothetical document
    hyde_prompt = f"""Write a detailed paragraph that would be a perfect excerpt from a
EU_regulation regulatory document answering this question. Write it as if you're quoting
directly from the regulation. Do not add any preamble.

Question: {query}

Regulatory excerpt:"""

    hypothetical_doc = llm.invoke(hyde_prompt).content
    print(f"  📝 HyDE generated ({len(hypothetical_doc)} chars): {hypothetical_doc[:120]}...")

    # Step 2: Retrieve using hypothetical doc as search query
    # PGVector will embed the hypothetical doc and find similar real docs
    hyde_results = vs.similarity_search(hypothetical_doc, k=k)
    print(f"  🔍 Retrieved {len(hyde_results)} docs using HyDE embedding")

    # Also get standard results for comparison logging
    standard_results = vs.similarity_search(query, k=k)

    # Check if different docs were retrieved
    hyde_contents = set(d.page_content[:80] for d in hyde_results)
    standard_contents = set(d.page_content[:80] for d in standard_results)
    overlap = hyde_contents & standard_contents
    print(f"  📊 Overlap with standard retrieval: {len(overlap)}/{k} docs")

    # Step 3: Rerank HyDE results (combine with Feature 2)
    pairs = [[query, doc.page_content] for doc in hyde_results]
    scores = reranker.predict(pairs)
    scored = sorted(zip(hyde_results, scores), key=lambda x: x[1], reverse=True)
    final_docs = [d for d, s in scored]

    # Step 4: Generate final answer
    context = "\n\n".join([doc.page_content for doc in final_docs])
    prompt = f"""You are a EU_regulation regulatory expert. Answer ONLY based on the provided context.
If the context doesn't contain the answer, say "I don't have enough information."

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt).content
    elapsed = round(time.time() - start, 2)

    return {
        "answer": response,
        "hypothetical_doc": hypothetical_doc[:200],
        "overlap_with_standard": len(overlap),
        "time": elapsed,
        "method": "hyde",
    }


# ── Compare ──
print("=" * 70)
print("📊 FEATURE 7: HyDE — Comparison")
print("=" * 70)

for q in TEST_QUESTIONS[:3]:
    print(f"\n{'='*70}")
    print(f"❓ {q}")

    print(f"\n🔹 BASELINE:")
    b = baseline_rag(q)
    print(f"   Answer: {b['answer'][:200]}...")

    print(f"\n🔸 HyDE:")
    h = rag_with_hyde(q)
    print(f"   HyDE doc: {h['hypothetical_doc'][:100]}...")
    print(f"   Overlap: {h['overlap_with_standard']}/3 docs same as baseline")
    print(f"   Time: {h['time']}s")
    print(f"   Answer: {h['answer'][:200]}...")

📊 FEATURE 7: HyDE — Comparison

❓ What is an AI system under the EU AI Act?

🔹 BASELINE:
Token not expired, using cached token...

Token not expired, using cached token...

   Answer: I don't have enough information. 

The provided context does not contain a definition of an AI system under the EU AI Act. It discusses the scope, application, and provisions of the Regulation, but do...

🔸 HyDE:
Token not expired, using cached token...

  📝 HyDE generated (1230 chars): "'AI system' means a software or program, including its data processing and generating algorithms, that is developed thr...
Token not expired, using cached token...

  🔍 Retrieved 3 docs using HyDE embedding
Token not expired, using cached token...

  📊 Overlap with standard retrieval: 0/3 docs
Token not expired, using cached token...

   HyDE doc: "'AI system' means a software or program, including its data processing and generating algorithms, t...
   Overlap: 0/3 docs same as baseline
   Time: 17.25s
   Answer: I don't 

In [236]:

def rag_with_multi_query(query, k=3, num_variations=3):
    """Generate query variations, retrieve for all, merge results."""
    start = time.time()

    # Step 1: Generate query variations
    variation_prompt = f"""Generate {num_variations} different versions of the following question
to retrieve relevant EU_regulation regulatory documents. Each variation should approach
the topic from a different angle. Return ONLY the questions, one per line.

Original: {query}

Variations:"""

    variations_text = llm.invoke(variation_prompt).content
    variations = [q.strip().lstrip("0123456789.-) ") for q in variations_text.strip().split("\n") if q.strip()]
    all_queries = [query] + variations[:num_variations]
    print(f"  📝 Generated {len(all_queries)} query variations:")
    for i, q in enumerate(all_queries):
        print(f"     {i+1}. {q[:80]}")

    # Step 2: Retrieve for each query
    all_docs = []
    seen_content = set()
    for q in all_queries:
        docs = vs.similarity_search(q, k=2)
        for doc in docs:
            content_hash = hash(doc.page_content[:100])
            if content_hash not in seen_content:
                seen_content.add(content_hash)
                all_docs.append(doc)

    print(f"  🔍 Retrieved {len(all_docs)} unique documents across all queries")

    # Step 3: Rerank merged results
    if len(all_docs) > k:
        pairs = [[query, doc.page_content] for doc in all_docs]
        scores = reranker.predict(pairs)
        scored = sorted(zip(all_docs, scores), key=lambda x: x[1], reverse=True)
        final_docs = [d for d, s in scored[:k]]
    else:
        final_docs = all_docs[:k]

    # Step 4: Generate
    context = "\n\n".join([doc.page_content for doc in final_docs])
    prompt = f"""You are a EU_regulation regulatory expert. Answer ONLY based on the provided context.
If the context doesn't contain the answer, say "I don't have enough information."

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt).content
    elapsed = round(time.time() - start, 2)

    return {
        "answer": response,
        "num_queries": len(all_queries),
        "unique_docs": len(all_docs),
        "time": elapsed,
        "method": "multi_query",
    }


# ── Test ──
print("=" * 70)
print("📊 Multi-Query Expansion — Comparison")
print("=" * 70)

q = TEST_QUESTIONS[2]  # "What rights does a data subject have regarding automated decision making?"
print(f"\n❓ {q}")

print(f"\n🔹 BASELINE:")
b = baseline_rag(q)
print(f"   Answer: {b['answer'][:200]}...")

print(f"\n🔸 MULTI-QUERY:")
mq = rag_with_multi_query(q)
print(f"   Queries: {mq['num_queries']} | Unique docs: {mq['unique_docs']}")
print(f"   Answer: {mq['answer'][:200]}...")

📊 Multi-Query Expansion — Comparison

❓ Who is considered a provider?

🔹 BASELINE:
Token not expired, using cached token...

Token not expired, using cached token...

   Answer: A 'provider' means a natural or legal person, public authority, agency or other body that develops an AI system or a general-purpose AI model or that has an AI system or a general-purpose AI model dev...

🔸 MULTI-QUERY:
Token not expired, using cached token...

  📝 Generated 4 query variations:
     1. Who is considered a provider?
     2. What entities are classified as providers under EU regulations?
     3. How does the EU define a provider in the context of regulatory compliance?
     4. Which organizations or individuals are deemed providers and subject to EU regula
Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

  🔍 Retrieved 6 unique documents across all queries
Token not expired, using

In [ ]:
# RAGA Evaluation

In [237]:
#! pip install datasets
from datasets import Dataset

In [238]:


# Since RAGAS by default uses OpenAI for evaluation,
# we'll use our LLM as the evaluator via a custom wrapper

def evaluate_rag_pipeline(pipeline_func, questions, ground_truths, pipeline_name="pipeline"):
    """
    Evaluate a RAG pipeline using manual faithfulness & relevancy checks.
    Works with Dell LLM (no OpenAI dependency).

    Metrics:
      - Faithfulness: Is the answer supported by the retrieved context?
      - Answer Relevancy: Does the answer address the question?
      - Context Relevancy: Are retrieved docs relevant to the question?
    """
    results = {
        "faithfulness_scores": [],
        "answer_relevancy_scores": [],
        "context_relevancy_scores": [],
    }

    for i, (question, ground_truth) in enumerate(zip(questions, ground_truths)):
        print(f"\n  📊 Evaluating Q{i+1}: {question[:60]}...")

        # Run the pipeline
        pipeline_result = pipeline_func(question)
        answer = pipeline_result["answer"]

        # Get context used
        context_docs = vs.similarity_search(question, k=3)
        context = "\n".join([d.page_content for d in context_docs])

        # ── Metric 1: Faithfulness ──
        faith_prompt = f"""On a scale of 0 to 1, rate how well the Answer is supported by the Context.
1.0 = fully supported, 0.0 = completely unsupported or fabricated.
Return ONLY a number between 0 and 1.

Context: {context[:2000]}

Answer: {answer[:1000]}

Score:"""
        faith_score = llm.invoke(faith_prompt).content.strip()
        try:
            faith_val = float(re.search(r'[0-9.]+', faith_score).group())
            faith_val = min(max(faith_val, 0), 1)
        except:
            faith_val = 0.5
        results["faithfulness_scores"].append(faith_val)

        # ── Metric 2: Answer Relevancy ──
        rel_prompt = f"""On a scale of 0 to 1, rate how well the Answer addresses the Question.
1.0 = perfectly relevant, 0.0 = completely irrelevant.
Return ONLY a number between 0 and 1.

Question: {question}

Answer: {answer[:1000]}

Score:"""
        rel_score = llm.invoke(rel_prompt).content.strip()
        try:
            rel_val = float(re.search(r'[0-9.]+', rel_score).group())
            rel_val = min(max(rel_val, 0), 1)
        except:
            rel_val = 0.5
        results["answer_relevancy_scores"].append(rel_val)

        # ── Metric 3: Context Relevancy ──
        ctx_prompt = f"""On a scale of 0 to 1, rate how relevant the retrieved Context is to the Question.
1.0 = perfectly relevant, 0.0 = completely irrelevant.
Return ONLY a number between 0 and 1.

Question: {question}

Context: {context[:2000]}

Score:"""
        ctx_score = llm.invoke(ctx_prompt).content.strip()
        try:
            ctx_val = float(re.search(r'[0-9.]+', ctx_score).group())
            ctx_val = min(max(ctx_val, 0), 1)
        except:
            ctx_val = 0.5
        results["context_relevancy_scores"].append(ctx_val)

        print(f"     Faith={faith_val:.2f} | Relevancy={rel_val:.2f} | Context={ctx_val:.2f}")

    # Compute averages
    avg_results = {
        "pipeline": pipeline_name,
        "faithfulness": round(np.mean(results["faithfulness_scores"]), 3),
        "answer_relevancy": round(np.mean(results["answer_relevancy_scores"]), 3),
        "context_relevancy": round(np.mean(results["context_relevancy_scores"]), 3),
    }

    print(f"\n  {'='*50}")
    print(f"  📊 {pipeline_name} — Average Scores:")
    print(f"     Faithfulness:      {avg_results['faithfulness']}")
    print(f"     Answer Relevancy:  {avg_results['answer_relevancy']}")
    print(f"     Context Relevancy: {avg_results['context_relevancy']}")

    return avg_results


# ── Run evaluation across pipelines ──
eval_questions = TEST_QUESTIONS[:3]
eval_ground_truths = [
    "EU_regulation requires parental consent for processing children's data under age 16, with member states able to lower to 13.",
    "Article 6 lists six lawful bases: consent, contract, legal obligation, vital interests, public task, legitimate interests.",
    "Article 22 gives data subjects the right not to be subject to automated decision-making including profiling.",
]

print("=" * 70)
print("📊 FEATURE 4: RAGAS-style Evaluation — All Pipelines")
print("=" * 70)

all_eval_results = []

# Evaluate Baseline
print("\n\n🔹 Evaluating: BASELINE")
r1 = evaluate_rag_pipeline(baseline_rag, eval_questions, eval_ground_truths, "Baseline")
all_eval_results.append(r1)

# Evaluate Reranked
print("\n\n🔹 Evaluating: RERANKED")
r2 = evaluate_rag_pipeline(rag_with_reranking, eval_questions, eval_ground_truths, "Reranked")
all_eval_results.append(r2)

# Evaluate Hybrid
print("\n\n🔹 Evaluating: HYBRID + RERANKED")
r3 = evaluate_rag_pipeline(rag_with_hybrid_search, eval_questions, eval_ground_truths, "Hybrid+Reranked")
all_eval_results.append(r3)

# Evaluate HyDE
print("\n\n🔹 Evaluating: HyDE")
r4 = evaluate_rag_pipeline(rag_with_hyde, eval_questions, eval_ground_truths, "HyDE")
all_eval_results.append(r4)

# ── Summary Table ──
print("\n\n" + "=" * 70)
print("📊 EVALUATION SUMMARY")
print("=" * 70)
print(f"{'Pipeline':<25} {'Faithfulness':<15} {'Answer Rel.':<15} {'Context Rel.':<15}")
print("-" * 70)
for r in all_eval_results:
    print(f"{r['pipeline']:<25} {r['faithfulness']:<15} {r['answer_relevancy']:<15} {r['context_relevancy']:<15}")

📊 FEATURE 4: RAGAS-style Evaluation — All Pipelines


🔹 Evaluating: BASELINE

  📊 Evaluating Q1: What is an AI system under the EU AI Act?...
Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

     Faith=0.00 | Relevancy=0.00 | Context=0.80

  📊 Evaluating Q2: Does the EU AI Act apply to companies outside the EU?...
Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

     Faith=1.00 | Relevancy=1.00 | Context=1.00

  📊 Evaluating Q3: Who is considered a provider?...
Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

To

In [ ]:
! pip install ragas

Defaulting to user installation because normal site-packages is not writeable
INFO: pip is looking at multiple versions of openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
INFO: pip is looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
  Using cached 

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
streamlit 1.40.1 requires protobuf<6,>=3.20, but you have protobuf 7.35.1 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [239]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    context_entity_recall,
    answer_similarity,
    answer_correctness,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

# ═══════════════════════════════════════════════════════════
# Wrap Dell LLM and Embeddings for RAGAS
# 'llm' is from build_llm() — already a ChatOpenAI instance [4]
# 'embeddings' is DSXEmbeddingInterface — already LangChain Embeddings [1]
# ═══════════════════════════════════════════════════════════

ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("✅ RAGAS configured with  LLM and  embeddings")

✅ RAGAS configured with  LLM and  embeddings


In [240]:
# ═══════════════════════════════════════════════════════════
# Prepare evaluation questions + ground truths
# ═══════════════════════════════════════════════════════════

eval_questions = [
    "What is the regulation policy for children under 10 Years?",
    "What are the conditions for lawful processing of personal data?",
    "What rights does a data subject have regarding automated decision making?",
    "What is the Processing of the national identification number?",
    "What are the penalties for EU_regulation violations?",
]

# Ground truth — what the correct answer should contain
eval_ground_truths = [
    "Under EU_regulation Article 8, where processing is based on consent, children below age 16 require parental consent. Member states may lower this to age 13 but not below.",
    "Article 6 of EU_regulation lists six lawful bases for processing: consent, contract performance, legal obligation, vital interests, public task, and legitimate interests.",
    "Article 22 gives data subjects the right not to be subject to decisions based solely on automated processing including profiling which produces legal or significant effects.",
    "Article 87 states that Member States may determine conditions for processing national identification numbers and shall ensure it is used under appropriate safeguards.",
    "Article 83 provides for administrative fines up to 20 million EUR or 4% of annual worldwide turnover, whichever is higher, for serious infringements.",
]

print(f"✅ Prepared {len(eval_questions)} evaluation questions")

✅ Prepared 5 evaluation questions


In [241]:
# ═══════════════════════════════════════════════════════════
# Generate answers using RAG pipeline
# Collect both answers and contexts for RAGAS
# ═══════════════════════════════════════════════════════════

eval_answers = []
eval_contexts = []

for i, question in enumerate(eval_questions):
    print(f"\n  📝 Generating answer for Q{i+1}: {question[:50]}...")

    # Retrieve context
    retrieved_docs = vs.similarity_search(question, k=3)
    context_list = [doc.page_content for doc in retrieved_docs]

    # Generate answer
    context_str = "\n\n".join(context_list)
    prompt = f"""You are a EU_regulation regulatory expert. Answer ONLY based on the provided context.
If the context doesn't contain the answer, say "I don't have enough information."

Context:
{context_str}

Question: {question}

Answer:"""

    answer = llm.invoke(prompt).content

    eval_answers.append(answer)
    eval_contexts.append(context_list)  # RAGAS expects List[str] per question

    print(f"     Answer: {answer[:150]}...")
    print(f"     Contexts: {len(context_list)} chunks retrieved")

print(f"\n✅ Generated {len(eval_answers)} answers")


  📝 Generating answer for Q1: What is the regulation policy for children under 1...
Token not expired, using cached token...

Token not expired, using cached token...

     Answer: I don't have enough information. 

The context mentions that the law on the protection of minors, namely persons below the age of 18, should not be af...
     Contexts: 3 chunks retrieved

  📝 Generating answer for Q2: What are the conditions for lawful processing of p...
Token not expired, using cached token...

Token not expired, using cached token...

     Answer: The conditions for lawful processing of personal data are outlined in Article 6 of the General Data Protection Regulation (GDPR) and include: (a) the ...
     Contexts: 0 chunks retrieved

  📝 Generating answer for Q3: What rights does a data subject have regarding aut...
Token not expired, using cached token...

Token not expired, using cached token...

     Answer: A data subject has the right to not be subject to a decision based solely on a

In [243]:
# ═══════════════════════════════════════════════════════════
# Build RAGAS Dataset
# ═══════════════════════════════════════════════════════════

ragas_data = {
    "question": eval_questions,
    "answer": eval_answers,
    "contexts": eval_contexts,          # List[List[str]]
    "ground_truth": eval_ground_truths,  # needed for recall & correctness
}

ragas_dataset = Dataset.from_dict(ragas_data)
print(f"✅ RAGAS dataset created: {ragas_dataset}")
print(ragas_dataset)

✅ RAGAS dataset created: Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 5
})
Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 5
})


In [245]:
# ═══════════════════════════════════════════════════════════
# Run RAGAS Evaluation — Direct Metrics
#
# Metrics:
#   faithfulness       — Is the answer grounded in the context?
#   answer_relevancy   — Does the answer address the question?
#   context_precision  — Are retrieved docs relevant?
#   context_recall     — Were all needed docs retrieved?
#   answer_correctness — Is the answer factually correct vs ground truth?
#   answer_similarity  — Semantic similarity to ground truth
# ═══════════════════════════════════════════════════════════

# Select metrics to evaluate
metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness,
    answer_similarity,
]

print("📊 Running RAGAS evaluation...")
print(f"   Metrics: {[m.name for m in metrics]}")
print(f"   Questions: {len(eval_questions)}")
print(f"   Using Dell LLM: llama-3-3-70b-instruct")
print(f"   Using Dell Embeddings: embedding gemma")
print("   This may take a few minutes...\n")

# Run evaluation with Dell LLM and Dell embeddings
result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("\n" + "=" * 60)
print("📊 RAGAS EVALUATION RESULTS")
print("=" * 60)
for metric_name, score in result.items():
    print(f"   {metric_name:<25} {score:.4f}")
print("=" * 60)

📊 Running RAGAS evaluation...
   Metrics: ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall', 'answer_correctness', 'answer_similarity']
   Questions: 5
   Using Dell LLM: llama-3-3-70b-instruct
   Using Dell Embeddings: embedding gemma
   This may take a few minutes...



Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...

Token not expired, using cached token...



Evaluating:  20%|██        | 6/30 [00:48<02:41,  6.74s/it]Exception raised in Job[2]: InternalServerError(Error code: 500 - {'message': 'An unexpected error occurred', 'request_id': '4cb5e74e2d9b8a4dd1c9fd67ac7e681a'})
Exception raised in Job[16]: InternalServerError(Error code: 500 - {'message': 'An unexpected error occurred', 'request_id': '3d0aa40221b6765544c02a481029497c'})
Exception raised in Job[3]: InternalServerError(Error code: 500 - {'message': 'An unexpected error occurred', 'request_id': 'a7d381722bc0cb68ccec903c4fccc122'})
Evaluating:  30%|███       | 9/30 [00:48<01:17,  3.69s/it]Exception raised in Job[4]: InternalServerError(Error code: 500 - {'message': 'An unexpected error occurred', 'request_id': '4189b62dc582274a68b982769c68c928'})
Exception raised in Job[15]: InternalServerError(Error code: 500 - {'message': 'An unexpected error occurred', 'request_id': '5228c81741eaa1591b0ce75c76c3e406'})
Exception raised in Job[12]: InternalServerError(Error code: 500 - {'message'

Token not expired, using cached token...

Token not expired, using cached token...



...

Evaluating:  70%|███████   | 21/30 [01:06<00:12,  1.35s/it]

Token not expired, using cached token...



Evaluating:  80%|████████  | 24/30 [01:23<00:17,  2.86s/it]Exception raised in Job[24]: InternalServerError(Error code: 500 - {'message': 'An unexpected error occurred', 'request_id': 'a15ec6223f694b3629daf65902e3cc58'})
Exception raised in Job[22]: InternalServerError(Error code: 500 - {'message': 'An unexpected error occurred', 'request_id': '4b5e3e4b8027c1573d621058ea9d352c'})
Exception raised in Job[21]: InternalServerError(Error code: 500 - {'message': 'An unexpected error occurred', 'request_id': '1ddf5a17e3a138725a2a064bb4f8ef36'})
Exception raised in Job[26]: InternalServerError(Error code: 500 - {'message': 'An unexpected error occurred', 'request_id': 'c86cb53409f64e6b5d63b94bd9839a34'})
Exception raised in Job[27]: InternalServerError(Error code: 500 - {'message': 'An unexpected error occurred', 'request_id': 'b6448b414583de374224e04d9e628a40'})
Evaluating: 100%|██████████| 30/30 [01:23<00:00,  2.79s/it]



📊 RAGAS EVALUATION RESULTS


AttributeError: 'EvaluationResult' object has no attribute 'items'

# Build a simple RAG (Baseline)
#### Add memory to the above RAG

In [246]:
from langchain.prompts import ChatPromptTemplate
from langchain.chains import RetrievalQA
from langchain.memory import ConversationBufferMemory


def rag_chain(query):
    # define RAG prompt
    prompt = ChatPromptTemplate.from_template(
            """You are an agent. Do not respond to questions which leads to controversy or racism or adultry question. 
            .For SSN related query, reply with "Sorry i cannot respond to this"
            Chat History: {chat_history}

            context: {context}

            Question: {question}

            Answer:
            """
    )
    # define memory tp store conversation history
    memory = ConversationBufferMemory(memory_key = 'chat_history', input_key = 'question')

    rag_chain = RetrievalQA.from_chain_type(
        llm = llm,
        retriever = retriever,
        chain_type_kwargs = {
            "prompt": prompt,
            "memory": memory
       }
    )

    return(rag_chain.invoke(query))



In [247]:
query = TEST_QUESTIONS[0]
rag_chain(query)

Generating new token...

Token not expired, using cached token...



{'query': 'What is an AI system under the EU AI Act?',
 'result': 'The EU AI Act does not provide a specific definition of an AI system. However, based on the context and the provisions outlined in the Act, an AI system can be understood as a software or program that uses artificial intelligence and machine learning to perform tasks, make decisions, or provide outputs. The Act applies to providers and deployers of AI systems, including those that place AI systems on the market, put them into service, or use them within the Union. The scope of the Act includes various types of AI systems, such as high-risk AI systems, general-purpose AI models, and AI systems used in different sectors and industries.'}

## Add Guardrails to ensure safety filters


# Tool using Agent (Agentic RAG)
    Retriever + Citation Checker + Summarizer

In [248]:
from langchain.agents import initialize_agent,Tool,AgentType

In [253]:

def context_retriever(text):
    retrieved_docs=vs.similarity_search(text,k=1)
    return ("\n\n".join([doc.page_content for doc in retrieved_docs]))

#Wrap function in tool
Tool_retriever=Tool(
    name="Context Retriever",
    func=context_retriever,
    description="retrieve regulations which are relavant for user's question/query."
)

#Citation Checker

def citation(text):
    llm_citation=get_dell_llm(temperature=0)
    citation_prompt="find if any citation is in place. Keep in mind that citation is not article or paragraph or chapter. It is basically any reference ",
    return llm_citation.predict(f"{citation_prompt} for given {text}")

Tool_citation=Tool(
    name="Citation Tool for text",
    func=citation,
    description="Provide citatiom for any given text"
)


def summarizer(text):
    llm_summary=get_dell_llm(temperature=0)
    return llm_summary.predict(f"Summarize specific answer within 2 sentence for given{text}")

#Wrap function in tool
Tool_summarizer=Tool(
    name="Answer Summarizer",
    func=summarizer,
    description="Summarize the data to make answer crisp and to the point"
)

In [254]:
memory=ConversationBufferMemory(memory_key="chat_history",return_messages=True)

llm=get_dell_llm(temperature=0)

agent=initialize_agent(
    tools=[Tool_retriever,Tool_citation,Tool_summarizer],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

In [255]:
query = TEST_QUESTIONS[1]

In [256]:
agent.run(query)



> Entering new AgentExecutor chain...
Token not expired, using cached token...

To answer whether the EU AI Act applies to companies outside the EU, I need to understand the scope and territorial application of the EU AI Act.

Action: Context Retriever
Action Input: EU AI Act territorial scopeToken not expired, using cached token...


Observation: (b) prohibitions of certain AI practices;
(c) specific requirements for high-risk AI systems and obligations for operators of such systems;
(d) harmonised transparency rules for certain AI systems;
(e) harmonised rules for the placing on the market of general-purpose AI models;
(f) rules on market monitoring, market surveillance, governance and enforcement;
(g) measures to support innovation, with a particular focus on SMEs, including start-ups.
Article 2
Scope
1.
This Regulation applies to:
(a) providers placing on the market or putting into service AI systems or placing on the market general-purpose AI models 
in the Union, irrespective o

'Agent stopped due to iteration limit or time limit.'

In [ ]:
x=input("enter query: ")
agent.run(x)

## 